# AWS S3 Data Intelligence with Cortex AI

Run cells in order. AWS setup (S3 bucket, IAM role, event notifications) must be done manually — see LAB_GUIDE.md.

---

## Step 1: Database, Schemas & Warehouse

In [ ]:
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE DATABASE HEALTHCARE_AI_DEMO
  COMMENT = 'Healthcare AI Intelligence Pipeline';

CREATE SCHEMA IF NOT EXISTS HEALTHCARE_AI_DEMO.RAW;
CREATE SCHEMA IF NOT EXISTS HEALTHCARE_AI_DEMO.PROCESSED;
CREATE SCHEMA IF NOT EXISTS HEALTHCARE_AI_DEMO.ANALYTICS;

CREATE OR REPLACE WAREHOUSE HEALTHCARE_AI_WH
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE;

USE WAREHOUSE HEALTHCARE_AI_WH;

## Step 2: Storage Integration

**Replace** `<YOUR_BUCKET_NAME>` and `<YOUR_AWS_IAM_ROLE_ARN>` below.

In [ ]:
CREATE OR REPLACE STORAGE INTEGRATION HEALTHCARE_S3_INTEGRATION
  TYPE = EXTERNAL_STAGE
  STORAGE_PROVIDER = 'S3'
  ENABLED = TRUE
  STORAGE_AWS_ROLE_ARN = '<YOUR_AWS_IAM_ROLE_ARN>'
  STORAGE_ALLOWED_LOCATIONS = ('s3://<YOUR_BUCKET_NAME>/healthcare/');

### Get Snowflake IAM User ARN & External ID

Copy `STORAGE_AWS_IAM_USER_ARN` and `STORAGE_AWS_EXTERNAL_ID` from the output.
Update your IAM role trust policy in AWS Console with these values.

In [ ]:
DESCRIBE INTEGRATION HEALTHCARE_S3_INTEGRATION;

### External Stages

**Replace** `<YOUR_BUCKET_NAME>`. Wait 30s after updating IAM trust policy.

In [ ]:
CREATE OR REPLACE STAGE HEALTHCARE_AI_DEMO.RAW.S3_MEDICAL_DOCS
  URL = 's3://<YOUR_BUCKET_NAME>/healthcare/pdfs/'
  STORAGE_INTEGRATION = HEALTHCARE_S3_INTEGRATION
  DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = TRUE);

CREATE OR REPLACE STAGE HEALTHCARE_AI_DEMO.RAW.S3_MEDICAL_TXT
  URL = 's3://<YOUR_BUCKET_NAME>/healthcare/txt/'
  STORAGE_INTEGRATION = HEALTHCARE_S3_INTEGRATION
  DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = TRUE);

CREATE OR REPLACE STAGE HEALTHCARE_AI_DEMO.RAW.S3_MEDICAL_AUDIO
  URL = 's3://<YOUR_BUCKET_NAME>/healthcare/audio/'
  STORAGE_INTEGRATION = HEALTHCARE_S3_INTEGRATION
  DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = TRUE);

CREATE OR REPLACE STAGE HEALTHCARE_AI_DEMO.ANALYTICS.SEMANTIC_MODELS
  DIRECTORY = (ENABLE = TRUE);

### Verify S3 Access

In [ ]:
LIST @HEALTHCARE_AI_DEMO.RAW.S3_MEDICAL_DOCS;

## Step 3: Snowpipes & Stream

In [ ]:
USE DATABASE HEALTHCARE_AI_DEMO;
USE SCHEMA RAW;

CREATE OR REPLACE TABLE RAW.FILES_LOG (
    FILE_ID       NUMBER AUTOINCREMENT PRIMARY KEY,
    FILE_NAME     VARCHAR NOT NULL,
    FILE_PATH     VARCHAR NOT NULL,
    FILE_TYPE     VARCHAR(10) NOT NULL,
    S3_EVENT_TIME TIMESTAMP_NTZ,
    LANDED_AT     TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    IS_PROCESSED  BOOLEAN DEFAULT FALSE,
    PROCESSED_AT  TIMESTAMP_NTZ
);

CREATE OR REPLACE FILE FORMAT RAW.METADATA_ONLY_FORMAT
  TYPE = 'CSV' RECORD_DELIMITER = NONE FIELD_DELIMITER = NONE;

CREATE OR REPLACE PIPE RAW.PIPE_MEDICAL_DOCS AUTO_INGEST = TRUE AS
  COPY INTO RAW.FILES_LOG (FILE_NAME, FILE_PATH, FILE_TYPE, S3_EVENT_TIME)
  FROM (
    SELECT METADATA$FILENAME, METADATA$FILENAME, 'PDF', METADATA$START_SCAN_TIME
    FROM @RAW.S3_MEDICAL_DOCS
  )
  FILE_FORMAT = (FORMAT_NAME = 'RAW.METADATA_ONLY_FORMAT');

CREATE OR REPLACE PIPE RAW.PIPE_MEDICAL_TXT AUTO_INGEST = TRUE AS
  COPY INTO RAW.FILES_LOG (FILE_NAME, FILE_PATH, FILE_TYPE, S3_EVENT_TIME)
  FROM (
    SELECT METADATA$FILENAME, METADATA$FILENAME, 'TXT', METADATA$START_SCAN_TIME
    FROM @RAW.S3_MEDICAL_TXT
  )
  FILE_FORMAT = (FORMAT_NAME = 'RAW.METADATA_ONLY_FORMAT');

CREATE OR REPLACE PIPE RAW.PIPE_MEDICAL_AUDIO AUTO_INGEST = TRUE AS
  COPY INTO RAW.FILES_LOG (FILE_NAME, FILE_PATH, FILE_TYPE, S3_EVENT_TIME)
  FROM (
    SELECT METADATA$FILENAME, METADATA$FILENAME,
      CASE WHEN METADATA$FILENAME ILIKE '%.mp3' THEN 'MP3' ELSE 'WAV' END,
      METADATA$START_SCAN_TIME
    FROM @RAW.S3_MEDICAL_AUDIO
  )
  FILE_FORMAT = (FORMAT_NAME = 'RAW.METADATA_ONLY_FORMAT');

CREATE OR REPLACE STREAM RAW.FILES_LOG_STREAM
  ON TABLE RAW.FILES_LOG APPEND_ONLY = TRUE;

### Get SQS Queue ARN

Copy `notification_channel` value for S3 event notification configuration.

In [ ]:
SHOW PIPES IN SCHEMA RAW;

## PAUSE: Configure S3 Event Notifications

Go to **AWS Console > S3 > Your bucket > Properties > Event notifications** and create 3 notifications pointing to the SQS ARN above:
- `snowpipe-pdfs` (prefix: `healthcare/pdfs/`)
- `snowpipe-txt` (prefix: `healthcare/txt/`)
- `snowpipe-audio` (prefix: `healthcare/audio/`)

Then continue below.

## Step 4: Ingest Files

In [ ]:
ALTER PIPE RAW.PIPE_MEDICAL_DOCS REFRESH;
ALTER PIPE RAW.PIPE_MEDICAL_TXT REFRESH;
ALTER PIPE RAW.PIPE_MEDICAL_AUDIO REFRESH;

Wait 1-2 minutes, then verify (expected: 6 PDF, 4 TXT, 5 WAV/MP3):

In [ ]:
SELECT FILE_TYPE, COUNT(*) AS FILES FROM RAW.FILES_LOG GROUP BY FILE_TYPE;

### Fix FILE_NAME prefix

In [ ]:
UPDATE RAW.FILES_LOG
SET FILE_NAME = REGEXP_REPLACE(FILE_NAME, '^healthcare/(pdfs|txt|audio)/', '')
WHERE FILE_NAME LIKE 'healthcare/%';

## Step 5: AI Processing Tables

In [ ]:
USE SCHEMA PROCESSED;

CREATE OR REPLACE TABLE PROCESSED.PDF_INTELLIGENCE (
    DOC_ID NUMBER AUTOINCREMENT PRIMARY KEY,
    FILE_ID NUMBER NOT NULL, FILE_NAME VARCHAR NOT NULL,
    PROCESSED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    PARSED_TEXT VARCHAR(16777216), PARSED_LAYOUT VARIANT,
    EXTRACTED_FIELDS VARIANT, DOC_CATEGORY VARCHAR(100),
    DOC_CATEGORY_CONFIDENCE FLOAT, SENTIMENT_SCORE FLOAT,
    SENTIMENT_DIMENSIONS VARIANT, SUMMARY VARCHAR(10000),
    DETECTED_LANGUAGE VARCHAR(50), TRANSLATED_TEXT VARCHAR(16777216),
    REDACTED_TEXT VARCHAR(16777216), KEY_INSIGHTS VARCHAR(10000),
    EMBEDDING VECTOR(FLOAT, 768)
);

CREATE OR REPLACE TABLE PROCESSED.TXT_INTELLIGENCE (
    TXT_ID NUMBER AUTOINCREMENT PRIMARY KEY,
    FILE_ID NUMBER NOT NULL, FILE_NAME VARCHAR NOT NULL,
    PROCESSED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    RAW_TEXT VARCHAR(16777216), EXTRACTED_FIELDS VARIANT,
    DOC_CATEGORY VARCHAR(100), DOC_CATEGORY_CONFIDENCE FLOAT,
    SENTIMENT_SCORE FLOAT, SENTIMENT_DIMENSIONS VARIANT,
    SUMMARY VARCHAR(10000), DETECTED_LANGUAGE VARCHAR(50),
    TRANSLATED_TEXT VARCHAR(16777216), REDACTED_TEXT VARCHAR(16777216),
    KEY_INSIGHTS VARCHAR(10000), EMBEDDING VECTOR(FLOAT, 768)
);

CREATE OR REPLACE TABLE PROCESSED.AUDIO_INTELLIGENCE (
    AUDIO_ID NUMBER AUTOINCREMENT PRIMARY KEY,
    FILE_ID NUMBER NOT NULL, FILE_NAME VARCHAR NOT NULL,
    PROCESSED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    TRANSCRIPT_TEXT VARCHAR(16777216), TRANSCRIPT_SEGMENTS VARIANT,
    AUDIO_DURATION_SECS FLOAT, SPEAKER_COUNT NUMBER,
    EXTRACTED_FIELDS VARIANT, CALL_CATEGORY VARCHAR(100),
    CALL_CATEGORY_CONFIDENCE FLOAT, SENTIMENT_SCORE FLOAT,
    SENTIMENT_DIMENSIONS VARIANT, SUMMARY VARCHAR(10000),
    DETECTED_LANGUAGE VARCHAR(50), TRANSLATED_TRANSCRIPT VARCHAR(16777216),
    CONSULTATION_NOTES VARCHAR(10000), EMBEDDING VECTOR(FLOAT, 768)
);

## Step 6: AI Processing Procedures

Run `assets/05_proc_pdf.sql`, `assets/06_proc_txt.sql`, `assets/07_proc_audio.sql` in a separate worksheet.

> If `claude-3-5-sonnet` is unavailable, replace with `mistral-large2`.

Then call:

In [ ]:
CALL PROCESSED.PROCESS_PDF_FILES();

In [ ]:
CALL PROCESSED.PROCESS_TXT_FILES();

In [ ]:
CALL PROCESSED.PROCESS_AUDIO_FILES();

In [ ]:
-- Verify
SELECT 'PDF' AS TYPE, COUNT(*) AS CNT FROM PROCESSED.PDF_INTELLIGENCE
UNION ALL SELECT 'TXT', COUNT(*) FROM PROCESSED.TXT_INTELLIGENCE
UNION ALL SELECT 'AUDIO', COUNT(*) FROM PROCESSED.AUDIO_INTELLIGENCE;

## Step 7: Cortex Search, Structured Data, Semantic View & Agent

Run in separate worksheets:
- `assets/11_cortex_search.sql`
- `assets/09_structured_data.sql`
- `assets/12_semantic_view.sql`
- `assets/13_cortex_agent.sql`

Verify:

In [ ]:
SHOW CORTEX SEARCH SERVICES IN DATABASE HEALTHCARE_AI_DEMO;

In [ ]:
DESCRIBE AGENT ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT;

## Step 8: Test the Agent

**Option A (recommended):** Go to **AI & ML > Intelligence > New Agent** > select `HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT`

**Option B:** SQL invocation:

In [ ]:
SELECT SNOWFLAKE.CORTEX.INVOKE_AGENT(
  'HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT',
  'Which providers have the highest total billed amounts?'
);

In [ ]:
SELECT SNOWFLAKE.CORTEX.INVOKE_AGENT(
  'HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT',
  'Find documents mentioning diabetes or hypertension'
);

In [ ]:
SELECT SNOWFLAKE.CORTEX.INVOKE_AGENT(
  'HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT',
  'What consultations discussed medication changes?'
);

## Pipeline Summary

In [ ]:
SELECT 'FILES_LOG' AS OBJECT, COUNT(*) AS ROW_COUNT FROM RAW.FILES_LOG
UNION ALL SELECT 'PDF_INTELLIGENCE', COUNT(*) FROM PROCESSED.PDF_INTELLIGENCE
UNION ALL SELECT 'TXT_INTELLIGENCE', COUNT(*) FROM PROCESSED.TXT_INTELLIGENCE
UNION ALL SELECT 'AUDIO_INTELLIGENCE', COUNT(*) FROM PROCESSED.AUDIO_INTELLIGENCE
UNION ALL SELECT 'PROVIDERS', COUNT(*) FROM ANALYTICS.PROVIDERS
UNION ALL SELECT 'PATIENTS', COUNT(*) FROM ANALYTICS.PATIENTS
UNION ALL SELECT 'CLAIMS', COUNT(*) FROM ANALYTICS.CLAIMS
UNION ALL SELECT 'APPOINTMENTS', COUNT(*) FROM ANALYTICS.APPOINTMENTS;

## Cleanup

In [ ]:
-- Uncomment to clean up:
-- DROP DATABASE IF EXISTS HEALTHCARE_AI_DEMO;
-- DROP WAREHOUSE IF EXISTS HEALTHCARE_AI_WH;
-- DROP STORAGE INTEGRATION IF EXISTS HEALTHCARE_S3_INTEGRATION;